In [ ]:
# setuptools>=69 ships a pkg_resources that works on Python 3.12
# (older versions use pkgutil.ImpImporter which was removed in 3.12)
!pip install -q "setuptools>=69.5.1"

# mmcv lite — pure-Python wheel, no compilation required, sufficient for inference
!pip install -q mmcv==1.7.2

# mmsegmentation 0.x — the API SegViT was written against
!pip install -q "mmsegmentation==0.30.0"

# SegViT extra deps
!pip install -q scipy timm

In [ ]:
import os, sys

if not os.path.exists('/content/SegVit'):
    !git clone https://github.com/zbwxp/SegVit.git /content/SegVit

%cd /content/SegVit
sys.path.insert(0, '/content/SegVit')

import torch
import mmseg
print(f"PyTorch:          {torch.__version__}")
print(f"MMSegmentation:   {mmseg.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('/content/checkpoints', exist_ok=True)
checkpoint_path = '/content/checkpoints/ade_51.3.pth'

if not os.path.exists(checkpoint_path):
    print("Downloading SegViT-Base checkpoint (ADE20K, 51.3 mIoU) ...")
    hf_hub_download(
        repo_id="Akide/SegViTv1",
        filename="ade_51.3.pth",
        local_dir="/content/checkpoints",
    )

print(f"Checkpoint ready — {os.path.getsize(checkpoint_path) / 1024**2:.0f} MB")

In [ ]:
import glob

print("Available SegViT configs:")
for p in sorted(glob.glob("configs/segvit/*.py")):
    print(" ", p)

In [ ]:
from mmseg.apis import init_segmentor
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

# SegViT-Base trained on ADE20K — change to segvit_vit-l_* for the larger model
config_file  = 'configs/segvit/segvit_vit-b_jax_640x640_160k_ade20k.py'
checkpoint_file = '/content/checkpoints/ade_51.3.pth'

model = init_segmentor(config_file, checkpoint_file, device=device)
print("Model loaded!")

In [ ]:
from mmseg.apis import inference_segmentor

# Download a sample image — replace img_path with your own file if preferred
img_path = '/content/sample.jpg'
if not os.path.exists(img_path):
    !wget -q -O /content/sample.jpg \
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikeroom.jpg/800px-Bikeroom.jpg"

result = inference_segmentor(model, img_path)
seg_map = result[0]

print(f"Input image:      {img_path}")
print(f"Segmentation map: shape={seg_map.shape}, dtype={seg_map.dtype}")
print(f"Classes detected: {sorted(set(seg_map.flatten().tolist()))}")

In [ ]:
from mmseg.apis import show_result_pyplot

# ADE20K class names (150 classes)
ADE20K_CLASSES = model.CLASSES

# Overlay the segmentation mask at 50 % opacity
show_result_pyplot(model, img_path, result, opacity=0.5)